# Heart Disease Classifier - Hackathon Showcase (CV + Ensemble)

This notebook runs the upgraded high-performance pipeline and generates presentation-ready outputs.

Pipeline highlights:
- 5-fold CV model selection
- Multi-model ensemble (DenseNet121, EfficientNet-B3, ConvNeXt-Tiny, ResNet18)
- TTA-enabled evaluation
- Leakage report + model comparison visuals

## 1) Runtime Setup (Colab GPU)

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Update if needed
%cd /content/drive/MyDrive/BYTE2BEAT
!pip install -q -r requirements.txt

## 2) Imports + Plot Style

In [ ]:
import os
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from IPython.display import display

PLOT_DIR = 'outputs/showcase_plots'
os.makedirs(PLOT_DIR, exist_ok=True)
sns.set_theme(style='whitegrid', context='talk')
plt.rcParams['figure.figsize'] = (10, 6)

print('Plot output folder:', PLOT_DIR)

## 3) Data Snapshot

In [ ]:
def get_split_counts(base='dataset_split'):
    rows = []
    for split in ['train', 'test']:
        split_dir = os.path.join(base, split)
        if not os.path.exists(split_dir):
            continue
        for cls in sorted(os.listdir(split_dir)):
            cls_dir = os.path.join(split_dir, cls)
            if os.path.isdir(cls_dir):
                cnt = sum(1 for f in os.listdir(cls_dir) if os.path.isfile(os.path.join(cls_dir, f)))
                rows.append({'split': split, 'class': cls, 'count': cnt})
    return pd.DataFrame(rows)

df_counts = get_split_counts()
display(df_counts.sort_values(['split', 'class']).reset_index(drop=True))

if not df_counts.empty:
    plt.figure(figsize=(11, 6))
    ax = sns.barplot(data=df_counts, x='class', y='count', hue='split', palette='Set2')
    ax.set_title('Dataset Class Distribution')
    plt.tight_layout()
    p = os.path.join(PLOT_DIR, 'dataset_class_distribution.png')
    plt.savefig(p, dpi=160)
    plt.show()
    print('Saved:', p)

## 4) Train (CV + Ensemble)
This can take substantial time on Colab because it trains multiple models with CV.

In [ ]:
!python train.py --config config.yaml

## 5) Evaluate Ensemble on Test Set (TTA from config)

In [ ]:
!python evaluate.py --config config.yaml

## 6) Core Test Metrics

In [ ]:
with open('outputs/metrics.json', 'r', encoding='utf-8') as f:
    metrics = json.load(f)

acc = float(metrics['accuracy'])
mf1 = float(metrics['macro_f1'])
print('Models:', metrics.get('models'))
print('TTA enabled:', metrics.get('tta'))
print(f'Accuracy: {acc:.4f}')
print(f'Macro F1: {mf1:.4f}')

report_df = pd.DataFrame(metrics['classification_report']).T
display(report_df.round(4))

plt.figure(figsize=(7, 4.5))
bars = plt.bar(['Accuracy', 'Macro F1'], [acc, mf1], color=['#1f77b4', '#ff7f0e'])
plt.ylim(0, 1)
plt.title('Ensemble Test Metrics')
for b in bars:
    plt.text(b.get_x() + b.get_width()/2, b.get_height() + 0.02, f'{b.get_height():.3f}', ha='center')
plt.tight_layout()
p = os.path.join(PLOT_DIR, 'ensemble_test_metrics.png')
plt.savefig(p, dpi=160)
plt.show()
print('Saved:', p)

In [ ]:
class_rows = [r for r in report_df.index if r not in ['accuracy', 'macro avg', 'weighted avg']]
if class_rows:
    df = report_df.loc[class_rows, ['f1-score']].reset_index().rename(columns={'index': 'class'})
    plt.figure(figsize=(8, 5))
    ax = sns.barplot(data=df, x='class', y='f1-score', palette='viridis')
    ax.set_ylim(0, 1)
    ax.set_title('Per-Class F1 (Ensemble)')
    for i, v in enumerate(df['f1-score']):
        ax.text(i, v + 0.02, f'{v:.3f}', ha='center')
    plt.tight_layout()
    p = os.path.join(PLOT_DIR, 'ensemble_per_class_f1.png')
    plt.savefig(p, dpi=160)
    plt.show()
    print('Saved:', p)

In [ ]:
cm_path = 'outputs/confusion_matrix.png'
if os.path.exists(cm_path):
    display(Image.open(cm_path))
else:
    print('Missing:', cm_path)

## 7) CV Summary + Model Ranking

In [ ]:
cv_path = 'outputs/cv_results.json'
if os.path.exists(cv_path):
    cv = json.load(open(cv_path, 'r', encoding='utf-8'))
    rows = []
    for m, stats in cv.items():
        rows.append({'model': m, 'cv_mean_macro_f1': stats['mean_macro_f1'], 'cv_std_macro_f1': stats['std_macro_f1']})
    cv_df = pd.DataFrame(rows).sort_values('cv_mean_macro_f1', ascending=False)
    display(cv_df.round(4))

    plt.figure(figsize=(10, 5))
    ax = sns.barplot(data=cv_df, x='model', y='cv_mean_macro_f1', palette='magma')
    ax.set_ylim(0, 1)
    ax.set_title('Cross-Validation Mean Macro-F1 by Model')
    for i, row in cv_df.reset_index(drop=True).iterrows():
        ax.text(i, row['cv_mean_macro_f1'] + 0.015, f"{row['cv_mean_macro_f1']:.3f}", ha='center')
    plt.tight_layout()
    p = os.path.join(PLOT_DIR, 'cv_model_comparison.png')
    plt.savefig(p, dpi=160)
    plt.show()
    print('Saved:', p)
else:
    print('Missing:', cv_path)

## 8) Per-Model Training Curves

In [ ]:
history_files = sorted([f for f in os.listdir('outputs') if f.endswith('_training_history.csv')])
print('History files:', history_files)

for hf in history_files:
    model = hf.replace('_training_history.csv', '')
    h = pd.read_csv(os.path.join('outputs', hf))

    plt.figure(figsize=(10, 4.5))
    plt.plot(h['epoch'], h['train_macro_f1'], marker='o', label='Train Macro F1')
    plt.plot(h['epoch'], h['val_macro_f1'], marker='o', label='Val Macro F1')
    plt.ylim(0, 1)
    plt.title(f'{model} Macro-F1 Curves')
    plt.xlabel('Epoch')
    plt.ylabel('Macro F1')
    plt.legend()
    plt.tight_layout()
    p = os.path.join(PLOT_DIR, f'{model}_macro_f1_curve.png')
    plt.savefig(p, dpi=160)
    plt.show()
    print('Saved:', p)

## 9) Leakage Report + Ensemble Manifest

In [ ]:
for f in ['outputs/leakage_report.json', 'models/ensemble_manifest.json', 'outputs/train_summary.json']:
    print('\n---', f, '---')
    if os.path.exists(f):
        data = json.load(open(f, 'r', encoding='utf-8'))
        if isinstance(data, dict) and len(data) > 8:
            small = {k: data[k] for k in list(data.keys())[:8]}
            display(small)
        else:
            display(data)
    else:
        print('Missing')

## 10) Sample Prediction Analysis

In [ ]:
pred_path = 'outputs/sample_preds.csv'
if os.path.exists(pred_path):
    sample_preds = pd.read_csv(pred_path)
    display(sample_preds)

    plt.figure(figsize=(9, 5))
    sns.histplot(sample_preds['confidence'], bins=10, kde=True, color='#2a9d8f')
    plt.title('Confidence Distribution (Sample Predictions)')
    plt.tight_layout()
    p = os.path.join(PLOT_DIR, 'sample_confidence_distribution.png')
    plt.savefig(p, dpi=160)
    plt.show()
    print('Saved:', p)
else:
    print('Missing:', pred_path)

## 11) Optional Single-Image Inference

In [ ]:
# Replace with real image
example_img = 'dataset_split/test/ASD/example.jpg'
!python predict.py --config config.yaml --checkpoint models/best_model.pth --image "$example_img" --topk 3

## 12) Saved Visual Assets

In [ ]:
saved = sorted([f for f in os.listdir(PLOT_DIR) if f.lower().endswith('.png')])
print('Saved plot files:')
for s in saved:
    print('-', os.path.join(PLOT_DIR, s))